# JC — Holy OG GPU Colab Factory

One-click entry point for `thebigostreets-byte/jc-the-holy-og`.

**Run order:** connect a GPU runtime, run the cells, authorize Google Drive when prompted, and JC stays synchronized with GitHub while persistent data lives in Drive.

The Colab GPU is for Python/AI/asset-processing workloads. The Three.js game still renders in the browser.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
JC_DRIVE='/content/drive/MyDrive/JC/Holy-OG'
for name in ['assets','checkpoints','outputs','logs']:
    os.makedirs(os.path.join(JC_DRIVE,name),exist_ok=True)
print('JC Drive workspace:',JC_DRIVE)

In [ ]:
# 2. Sync the real JC repository into Drive
import os,subprocess
REPO='https://github.com/thebigostreets-byte/jc-the-holy-og.git'
REPO_DIR=os.path.join(JC_DRIVE,'repo')
if not os.path.exists(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','--branch','main',REPO,REPO_DIR],check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',REPO_DIR,'checkout','main'],check=True)
    subprocess.run(['git','-C',REPO_DIR,'reset','--hard','origin/main'],check=True)
print('JC commit:',subprocess.check_output(['git','-C',REPO_DIR,'log','-1','--oneline'],text=True).strip())

In [ ]:
# 3. GPU check
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi'],text=True))
except Exception as e:
    raise RuntimeError('No NVIDIA GPU runtime. In Colab choose Runtime > Change runtime type > GPU, then rerun.') from e

In [ ]:
# 4. Install JC asset tooling — hardened Colab setup
import os,subprocess,sys,shutil
subprocess.run([sys.executable,'-m','pip','install','-U','pip','setuptools','wheel'],check=True)
subprocess.run([sys.executable,'-m','pip','install','numpy','pillow','trimesh','pygltflib'],check=True)
if os.path.exists(os.path.join(REPO_DIR,'package.json')):
    NODE_DIR='/content/jc-node'
    if os.path.exists(NODE_DIR): shutil.rmtree(NODE_DIR,ignore_errors=True)
    os.makedirs(NODE_DIR,exist_ok=True)
    shutil.copy2(os.path.join(REPO_DIR,'package.json'),os.path.join(NODE_DIR,'package.json'))
    print('Node:',subprocess.check_output(['node','--version'],text=True).strip())
    print('npm:',subprocess.check_output(['npm','--version'],text=True).strip())
    result=subprocess.run(['npm','install','--no-package-lock','--no-audit','--no-fund'],cwd=NODE_DIR,text=True,capture_output=True)
    print(result.stdout[-4000:])
    if result.returncode:
        print(result.stderr[-8000:])
        raise RuntimeError('npm install failed in the temporary Colab VM workspace. The full npm error is printed above.')
    print('Node tooling ready:',NODE_DIR)
print('JC asset tooling ready in Colab.')


In [ ]:
# 5. Verify CUDA — automatic GPU diagnostics
import torch,subprocess
print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
print('CUDA build:',torch.version.cuda)
try:
    smi=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True).strip()
    print('nvidia-smi:',smi)
except Exception as e: print('nvidia-smi check failed:',e)
if not torch.cuda.is_available(): raise RuntimeError('CUDA is not available. In Colab select Runtime > Change runtime type > GPU, then Runtime > Restart session and run all cells again.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
print('JC GPU READY')

## Drive layout

`My Drive/JC/Holy-OG/`

- `repo/` — synchronized JC source
- `assets/` — generated assets
- `checkpoints/` — model/checkpoint files
- `outputs/` — generated outputs
- `logs/` — logs

Python packages use Colab's managed runtime. Node/Vite packages use the temporary `/content/jc-node` VM workspace, so Google Drive does not need `node_modules`.

In [ ]:
# 7. Final JC status
import os,subprocess
print('Repo:',REPO_DIR)
print('Commit:',subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],text=True).strip())
print('Drive workspace exists:',os.path.isdir(JC_DRIVE))
print('JC COLAB FACTORY READY')

## 6. JC Gradio Control Center
\nThe Colab GPU is controlled through a Gradio UI. The Three.js JC game remains the browser client.

In [ ]:
# 6. Launch the JC Gradio Control Center
import os,sys,subprocess,shutil,time
subprocess.run([sys.executable,'-m','pip','install','-q','gradio'],check=True)
import gradio as gr

def cmd(args):
    p=subprocess.run(args,cwd=REPO_DIR,text=True,capture_output=True)
    return (p.stdout or '') + (p.stderr or '')

def status():
    commit=subprocess.check_output(['git','-C',REPO_DIR,'log','-1','--oneline'],text=True).strip()
    gpu=cmd(['nvidia-smi','--query-gpu=name,memory.total,memory.used,utilization.gpu','--format=csv,noheader']).strip()
    return f'REPO\n{commit}\n\nGPU\n{gpu}\n\nDRIVE\n{JC_DRIVE}'

def sync_repo():
    return cmd(['git','-C',REPO_DIR,'fetch','origin','main']) + cmd(['git','-C',REPO_DIR,'reset','--hard','origin/main'])

def build_site():
    return cmd(['npm','run','build','--prefix','/content/jc-node'])

def verify_assets():
    return cmd(['npm','run','verify:assets','--prefix','/content/jc-node'])

with gr.Blocks(title='JC — Holy OG GPU Control Center') as demo:
    gr.Markdown('# JC — Holy OG GPU Control Center')
    gr.Markdown('GPU asset factory + repository control panel. The Three.js game runs in the browser.')
    with gr.Row():
        status_btn=gr.Button('Refresh Status',variant='primary')
        sync_btn=gr.Button('Sync GitHub')
        build_btn=gr.Button('Build JC')
        verify_btn=gr.Button('Verify Assets')
    status_box=gr.Textbox(label='JC Status',lines=8)
    log_box=gr.Textbox(label='Command Output',lines=16)
    status_btn.click(status,outputs=status_box)
    sync_btn.click(sync_repo,outputs=log_box)
    build_btn.click(build_site,outputs=log_box)
    verify_btn.click(verify_assets,outputs=log_box)
    demo.load(status,outputs=status_box)

print('Starting Gradio...')
demo.launch(share=True,debug=True)


In [ ]:
# 7. Final JC status
import os,subprocess
print('Repo:',REPO_DIR)
print('Commit:',subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],text=True).strip())
print('Drive workspace exists:',os.path.isdir(JC_DRIVE))
print('JC COLAB FACTORY READY — GRADIO UI')
